# 03 — Compute Author Seniority from Google Scholar

A common metric in scientometrics for operationalising career stage is the number of **consecutive years** an author has published up to a reference year. Authors who have published every year for many years are classified as senior; those with only a few recent years are early-career.

This notebook shows how to:
1. Retrieve an author's article list from Google Scholar
2. Parse publication years
3. Compute consecutive publishing years up to a reference year
4. Classify authors into early / mid / senior career stages
5. Visualise the publication timeline

This logic mirrors the core of the `inspiration/EDA.ipynb` notebook.

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from serpapi import GoogleSearch

In [ ]:
load_dotenv('../.env')
SERPAPI_KEY = os.getenv('SERPAPI_KEY')
if not SERPAPI_KEY or SERPAPI_KEY == 'your_serpapi_key_here':
    raise ValueError('SERPAPI_KEY not set.')

## 1. Helper functions

In [ ]:
def get_author_articles(author_id: str, api_key: str) -> list:
    """Retrieve all articles for a Google Scholar author."""
    all_articles = []
    start = 0
    while True:
        params = {
            'engine': 'google_scholar_author',
            'author_id': author_id,
            'sort': 'pubdate',
            'num': 100,
            'start': start,
            'api_key': api_key,
        }
        results = GoogleSearch(params).get_dict()
        page = results.get('articles', [])
        if not page:
            break
        all_articles.extend(page)
        if len(page) < 100:
            break
        start += 100
        time.sleep(0.5)
    return all_articles


def consecutive_publishing_years(pub_years: list, end_year: int) -> int:
    """
    Count consecutive years with at least one publication,
    counting backwards from end_year (inclusive).
    """
    years_with_pubs = set(int(y) for y in pub_years if y is not None)
    consecutive = 0
    year = end_year
    while year in years_with_pubs:
        consecutive += 1
        year -= 1
    return consecutive


def classify_seniority(consecutive_years: int) -> str:
    """Classify seniority based on consecutive publishing years."""
    if consecutive_years < 3:
        return 'early'
    elif consecutive_years <= 8:
        return 'mid'
    else:
        return 'senior'

## 2. Fetch articles for an example author

In [ ]:
# Replace with any Google Scholar author ID
AUTHOR_ID = 'hkFGKj8AAAAJ'
REFERENCE_YEAR = 2023

articles = get_author_articles(AUTHOR_ID, SERPAPI_KEY)
print(f'Fetched {len(articles)} articles')

In [ ]:
# Parse publication years (some articles may have no year)
pub_years = []
for a in articles:
    year_raw = a.get('year')
    if year_raw:
        try:
            pub_years.append(int(year_raw))
        except ValueError:
            pass

print(f'Articles with a year: {len(pub_years)}')
print(f'Year range: {min(pub_years)} – {max(pub_years)}')

## 3. Compute seniority

In [ ]:
consec = consecutive_publishing_years(pub_years, REFERENCE_YEAR)
seniority = classify_seniority(consec)

print(f'Consecutive publishing years (up to {REFERENCE_YEAR}): {consec}')
print(f'Seniority classification: {seniority}')

## 4. Visualise publication timeline

In [ ]:
from collections import Counter

year_counts = Counter(pub_years)
years_sorted = sorted(year_counts.keys())
counts = [year_counts[y] for y in years_sorted]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(years_sorted, counts, color='steelblue')
ax.axvline(REFERENCE_YEAR, color='red', linestyle='--', label=f'Reference year ({REFERENCE_YEAR})')
ax.set_xlabel('Publication year')
ax.set_ylabel('Number of publications')
ax.set_title(f'Publication timeline\nSeniority: {seniority} ({consec} consecutive years up to {REFERENCE_YEAR})')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Try different reference years

The seniority classification depends on the reference year. Here we show how it changes.

In [ ]:
print(f'Seniority at different reference years:')
for ref_year in range(2018, 2025):
    c = consecutive_publishing_years(pub_years, ref_year)
    s = classify_seniority(c)
    print(f'  {ref_year}: {c:2d} consecutive years → {s}')